## Root Cause Analysis - Exp2

* Goals
  * Root Cause Analysis on individual RAG results --> 2 RAGs results comparison
  * Provide actionable suggestions
* About Exp2
  * Build Langgraph Agent System for Root Cause Analysis

In [2]:
%load_ext autoreload
%autoreload 2

import os
from sqlalchemy import make_url
import pandas as pd

from utils_root_cause import *

from langchain_groq import ChatGroq
from typing import TypedDict, Literal, Optional
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, SystemMessage
import json

import warnings
warnings.filterwarnings('ignore')


exp_llm = ChatGroq(
    groq_api_key=os.environ["GROQ_TOKEN"],
    model_name="openai/gpt-oss-20b", 
    temperature=0.78
)

DATABASE_URL_PUBLIC = os.getenv("DATABASE_URL_PUBLIC_RAG_DR")
DATABASE_URL = DATABASE_URL_PUBLIC.replace("postgres://", "postgresql://")
db_url = make_url(DATABASE_URL)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
auto_eval_output = get_auto_eval_output(db_url)
print(auto_eval_output.shape)

auto_eval_output.head(n=2)

(60, 16)


,config_hash,dataset,embedding_model,top_n_retrieval,semantic_weight,answer_gen_llm,query,context,retrieved_content,same_context,retrieval_quality_score,rq_reasoning,referenced_answer,ai_answer,answer_quality_score,aq_reasoning
0,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,Just have the associate sign the back and then...,true,3,The retrieved content reproduces the entire co...,Have the check reissued to the proper payee.Ju...,To deposit a cheque issued to an associate in ...,2,The AI’s answer correctly identifies that the ...
1,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,true,3,The retrieved content matches the context exac...,Sure you can. You can fill in whatever you wa...,"Yes, you can fill in your business name and ad...",2,The AI’s answer correctly states that a busine...


In [ ]:
# =========================
# State Schema
# =========================

class RCAState(TypedDict):
    query: str
    retrieved_content: str
    referenced_answer: str
    ai_answer: str

    retrieval_analysis: Optional[str]
    answer_analysis: Optional[str]
    improvement_plan: Optional[str]

    next_agent: Optional[str]